<a href="https://colab.research.google.com/github/yadavrishikesh/Masterclass-V2-2026/blob/main/module5/code/CaseStudy_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌏 Case Study Assignment: MKB–UNet–LSTM for Climate Prediction
## Building Blocks of Deep Learning for Aerosol–Temperature Forecasting over South Asia

**Total Duration:** 60 minutes  
**Question 1 (Spatial Pipeline):** 30 minutes  
**Question 2 (Temporal Pipeline):** 30 minutes  

---

### Architecture Overview
```
Input Climate Maps (BC_AOD, DU_AOD, SU_AOD, T2M)
        ↓
[Multi-Kernel Block (MKB)]   ← captures features at 1×1, 3×3, 5×5 scales
        ↓
[U-Net Encoder → Bottleneck → Decoder]  ← spatial context + skip connections
        ↓
[LSTM Temporal Module]       ← learns month-to-month patterns
        ↓
Predicted T2M (Next Month)
```

### Dataset
- **Variables:** BC_AOD (Black Carbon), DU_AOD (Dust), SU_AOD (Sulfate), T2M (2-m Temperature)
- **Region:** South Asia (28°N–37°N, 70°E–81.25°E)
- **Period:** January 1980 – December 2023 (528 monthly time steps)
- **Grid:** 19 × 19 spatial points

### Instructions
- Fill in all `_______` blanks and `# YOUR CODE HERE` sections
- Run each cell after completing it
- Answer the interpretation questions in markdown cells
- **Do not skip cells** — later parts depend on earlier ones


## ⚙️ Setup — Run This First

In [ ]:
# Run this cell first — installs and imports everything needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

print("✅ All libraries loaded successfully!")
print(f"PyTorch version: {torch.__version__}")


---
# 🗺️ Question 1: Building the Spatial Learning Pipeline
**⏱ Estimated Time: 30 minutes**

In this question you will explore the dataset, then implement — step by step — the spatial components of the MKB–UNet architecture.


## Part A: Dataset Exploration (5 min)

The dataset contains four CSV files, each shaped as `(361 grid cells × 528 months)` plus `lat` and `lon` columns.

**Your task:** Complete the data-loading code, print dimensions, and visualize one aerosol field.


In [ ]:
# ── Part A.1: Load all four CSV files ──────────────────────────────────────
DATA_DIR = "data/data/"   # adjust path if needed

bc_df  = pd.read_csv(_______ + "BC_AOD_time_series.csv")
du_df  = pd.read_csv(_______ + "_______")
su_df  = pd.read_csv(_______ + "_______")
t2m_df = pd.read_csv(_______ + "_______")

print("=== BC_AOD DataFrame shape:", bc_df.shape)
print("Columns (first 5):", list(bc_df.columns[:5]))
print("Columns (last 3) :", list(bc_df.columns[-3:]))


In [ ]:
# ── Part A.2: Detect spatial and temporal dimensions ───────────────────────
lats      = sorted(bc_df['lat'].unique())
lons      = sorted(bc_df['lon'].unique())
time_cols = [c for c in bc_df.columns if c not in ['lat', 'lon']]

n_lat  = len(lats)
n_lon  = len(lons)
n_time = _______          # fill in: length of time_cols

print(f"Latitude  range : {lats[0]}°N  →  {lats[-1]}°N  ({n_lat} points)")
print(f"Longitude range : {lons[0]}°E  →  {lons[-1]}°E  ({n_lon} points)")
print(f"Time steps      : {time_cols[0]}  →  {time_cols[-1]}  ({n_time} months)")
print(f"Variables       : BC_AOD, DU_AOD, SU_AOD, T2M")


In [ ]:
# ── Part A.3: Reshape flat CSV → 3-D arrays (time × lat × lon) ────────────
def csv_to_3d(df, lats, lons, time_cols):
    """Pivot a flat (grid_cell × time) DataFrame into (time, lat, lon) array."""
    n_t, n_la, n_lo = len(time_cols), len(lats), len(lons)
    arr = np.zeros((n_t, n_la, n_lo))
    for i, la in enumerate(lats):
        for j, lo in enumerate(lons):
            row = df[(df['lat'] == la) & (df['lon'] == lo)]
            if len(row):
                arr[:, i, j] = row[time_cols].values[0]
    return arr

bc_arr  = csv_to_3d(bc_df,  lats, lons, time_cols)   # shape: (528, 19, 19)
du_arr  = csv_to_3d(du_df,  lats, lons, time_cols)
su_arr  = csv_to_3d(su_df,  lats, lons, time_cols)
t2m_arr = csv_to_3d(t2m_df, lats, lons, time_cols)

print("bc_arr  shape:", bc_arr.shape)
print("du_arr  shape:", _______.shape)   # fill in
print("su_arr  shape:", _______.shape)
print("t2m_arr shape:", _______.shape)


In [ ]:
# ── Part A.4: Visualize one spatial snapshot ────────────────────────────────
month_idx = 0   # January 1980

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

data_list  = [bc_arr, du_arr, su_arr, t2m_arr]
titles     = ['BC_AOD', 'DU_AOD', 'SU_AOD', 'T2M (°C)']
cmaps      = ['YlOrRd', 'YlOrBr', 'Blues', 'RdYlBu_r']

for ax, data, title, cmap in zip(axes, data_list, titles, cmaps):
    im = ax.imshow(_______, origin='lower',   # fill in: which slice to plot?
                   cmap=cmap, aspect='auto')
    ax.set_title(f'{title}\n{time_cols[month_idx]}')
    ax.set_xlabel('Longitude index')
    ax.set_ylabel('Latitude index')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('South Asia Climate Fields — January 1980', fontsize=13)
plt.tight_layout()
plt.show()


### 📝 Interpretation — Part A

Answer these questions in this cell:

1. **What are the spatial dimensions of our dataset?** (rows × columns)
2. **Which variable shows the strongest spatial gradient** in January 1980?
3. **Why do we reshape the CSV into a 3-D array** (time × lat × lon) before feeding it into a CNN?

*Your answers here:*


---
## Part B: Implement a 3×3 Convolution Filter (5 min)

Convolution is the core operation in both the MKB and U-Net. Here you apply a single 3×3 filter to one aerosol map and visualize the edge-detection effect.


In [ ]:
# ── Part B: Apply a hand-crafted 3×3 Sobel edge filter to BC_AOD ──────────
# Select one spatial snapshot
snapshot = bc_arr[0]   # shape: (19, 19)

# Convert to tensor: (batch, channels, H, W)
x = torch.tensor(snapshot, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
print("Input tensor shape:", x.shape)

# Define a 3×3 Sobel-like edge-detection filter
edge_filter = torch.tensor([[[-1, -1, -1],
                              [-1,  8, -1],
                              [-1, -1, -1]]], dtype=torch.float32).unsqueeze(0)

# Apply convolution  (padding=1 keeps the spatial size the same)
conv_layer = nn.Conv2d(in_channels=_______, out_channels=1,
                       kernel_size=3, padding=1, bias=False)

# Copy our hand-crafted weights into the layer
with torch.no_grad():
    conv_layer.weight = nn.Parameter(edge_filter)

# Forward pass
with torch.no_grad():
    output = conv_layer(_______)   # fill in: what do we pass?

print("Output tensor shape:", output.shape)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(snapshot, cmap='YlOrRd', origin='lower')
axes[0].set_title('Original BC_AOD (Jan 1980)')
axes[1].imshow(output.squeeze().numpy(), cmap='seismic', origin='lower')
axes[1].set_title('After 3×3 Edge Filter')
for ax in axes:
    ax.set_xlabel('Longitude index'); ax.set_ylabel('Latitude index')
plt.tight_layout(); plt.show()


### 📝 Interpretation — Part B

1. **What does the edge filter highlight?** Where are the strongest responses?
2. **Why does `padding=1` keep the output the same size as the input?**
3. In the MKB architecture, filters are *learned* rather than hand-crafted. What advantage does that give?

*Your answers here:*


---
## Part C: Implement a Multi-Kernel Block (MKB) (7 min)

The MKB applies convolutions at three scales simultaneously (1×1, 3×3, 5×5) and concatenates the results.  
This lets the network capture fine local patterns *and* coarser regional patterns at the same time.

```
Input
  ├── 1×1 Conv  → feature map A
  ├── 3×3 Conv  → feature map B
  └── 5×5 Conv  → feature map C
         ↓
  Concatenate [A, B, C] along channel dimension
```


In [ ]:
# ── Part C: Complete the Multi-Kernel Block ─────────────────────────────────

class MultiKernelBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Branch 1: 1×1 convolution — captures pointwise (local) features
        self.k1 = nn.Conv2d(in_channels, out_channels,
                            kernel_size=1, padding=0)

        # Branch 2: 3×3 convolution — captures neighbourhood features
        self.k3 = nn.Conv2d(_______, _______,          # fill in
                            kernel_size=3, padding=1)

        # Branch 3: 5×5 convolution — captures broader regional features
        self.k5 = _______                              # fill in (use kernel_size=5, padding=2)

        self.relu = nn.ReLU()

    def forward(self, x):
        # Apply each branch
        out1 = self.relu(self.k1(x))
        out3 = self.relu(self.k3(_____))               # fill in
        out5 = _______                                 # fill in

        # Concatenate along the channel dimension (dim=1)
        fused = torch.cat([out1, out3, _______], dim=1)  # fill in
        return fused

# ── Test the block ────────────────────────────────────────────────────────────
# Stack all 4 variables into a (batch=1, channels=4, H=19, W=19) tensor
all_vars = np.stack([bc_arr[0], du_arr[0], su_arr[0], t2m_arr[0]], axis=0)
x_test   = torch.tensor(all_vars, dtype=torch.float32).unsqueeze(0)
print("Input shape:", x_test.shape)

mkb = MultiKernelBlock(in_channels=4, out_channels=8)
with torch.no_grad():
    mkb_out = mkb(x_test)

print("MKB output shape:", mkb_out.shape)
print(f"  → {mkb_out.shape[1]} channels  (8 from each of the 3 branches = {8*3} total)")


In [ ]:
# ── Visualize MKB feature maps ───────────────────────────────────────────────
n_show = 6
fig, axes = plt.subplots(1, n_show, figsize=(16, 3))

for i, ax in enumerate(axes):
    ax.imshow(mkb_out[0, i].numpy(), cmap='viridis', origin='lower')
    ax.set_title(f'Channel {i}')
    ax.axis('off')

plt.suptitle('MKB Feature Maps (first 6 of 24 channels)', fontsize=12)
plt.tight_layout()
plt.show()


### 📝 Interpretation — Part C

1. **Why does the output have 24 channels** when each branch produces 8?
2. **Look at the feature maps.** Do they all look the same? What differences do you notice between channels from the 1×1 and 5×5 branches?
3. **In climate science terms:** what kind of climate patterns would a 5×5 kernel capture that a 1×1 kernel would miss?

*Your answers here:*


---
## Part D: Build a Mini Encoder (5 min)

The encoder progressively **downsamples** the spatial maps while increasing the number of feature channels.  
Each encoder stage = `Conv2d → ReLU → MaxPool`.


In [ ]:
# ── Part D: Complete the Mini Encoder ───────────────────────────────────────

class MiniEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Stage 1: 4 → 16 channels,  no spatial downsampling yet
        self.conv1 = nn.Conv2d(4, 16, kernel_size=3, padding=1)

        # Stage 2: 16 → 32 channels
        self.conv2 = nn.Conv2d(_______, 32, kernel_size=3, padding=1)  # fill in

        # Max-pooling — halves H and W
        self.pool  = nn.MaxPool2d(kernel_size=_______, stride=2)       # fill in

        self.relu  = nn.ReLU()

    def forward(self, x):
        # Stage 1
        e1 = self.relu(self.conv1(x))     # shape: (B, 16, 19, 19)

        # Stage 2 + pooling
        e2 = self.relu(self.conv2(e1))    # shape: (B, 32, 19, 19)
        e2_pool = self.pool(_______)      # fill in — what gets pooled?

        return e1, e2, e2_pool            # return all for skip connections

# ── Test the encoder ──────────────────────────────────────────────────────────
encoder = MiniEncoder()
with torch.no_grad():
    e1, e2, e2_pool = encoder(x_test)

print("Encoder output shapes:")
print(f"  e1      (stage 1, before pool) : {e1.shape}")
print(f"  e2      (stage 2, before pool) : {e2.shape}")
print(f"  e2_pool (stage 2, after pool)  : {e2_pool.shape}")


### 📝 Interpretation — Part D

1. **How did the spatial dimensions change** after MaxPool2d?  Fill the table:

| Stage | Shape | H × W |
|-------|-------|--------|
| Input | (1, 4, 19, 19) | 19 × 19 |
| After Stage 1 Conv | ? | ? |
| After Stage 2 Conv | ? | ? |
| After MaxPool | ? | ? |

2. **Why does the encoder reduce spatial size?** What is the computational benefit?
3. **Why do we save `e1` and `e2`** even before pooling? (Hint: think about Part E.)

*Your answers here:*


---
## Part E: Build a Mini Decoder with Skip Connections (5 min)

The decoder **upsamples** back to the original resolution and **concatenates skip connections** from the encoder.  
Skip connections let the decoder recover fine spatial details that were lost during downsampling.


In [ ]:
# ── Part E: Complete the Mini Decoder ───────────────────────────────────────

class MiniDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Upsample: doubles H and W
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear',
                                    align_corners=False)

        # After concatenating upsampled (32ch) + skip e2 (32ch) → 64 input channels
        self.conv_up  = nn.Conv2d(64, 32, kernel_size=3, padding=1)

        # Final output: map to 1 channel (temperature prediction)
        self.conv_out = nn.Conv2d(_______, 1, kernel_size=1)   # fill in

        self.relu = nn.ReLU()

    def forward(self, bottleneck, skip_e2):
        # Step 1: Upsample the bottleneck
        up = self.upsample(_______)                            # fill in

        # Step 2: Concatenate with skip connection along channel dim
        cat = torch.cat([up, _______], dim=1)                 # fill in (which skip?)

        # Step 3: Convolve + activate
        dec = self.relu(self.conv_up(cat))

        # Step 4: Project to 1 output channel
        out = self.conv_out(_______)                           # fill in
        return out

# ── Test the decoder ──────────────────────────────────────────────────────────
decoder = MiniDecoder()
with torch.no_grad():
    dec_out = decoder(e2_pool, e2)

print("Decoder output shape:", dec_out.shape)

# Visualize encoder vs decoder features
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(e2_pool[0, 0].numpy(), cmap='plasma', origin='lower')
axes[0].set_title(f'Bottleneck feature\n{e2_pool.shape[-2]}×{e2_pool.shape[-1]}')

axes[1].imshow(e2[0, 0].numpy(), cmap='plasma', origin='lower')
axes[1].set_title(f'Skip connection (e2)\n{e2.shape[-2]}×{e2.shape[-1]}')

axes[2].imshow(dec_out[0, 0].detach().numpy(), cmap='RdYlBu_r', origin='lower')
axes[2].set_title(f'Decoder output\n{dec_out.shape[-2]}×{dec_out.shape[-1]}')

for ax in axes: ax.axis('off')
plt.suptitle('Encoder Bottleneck → Skip → Decoder Output', fontsize=12)
plt.tight_layout(); plt.show()


### 📝 Interpretation — Part E

1. **What happened to the spatial size** after upsampling?
2. **Why do we concatenate the skip connection *before* applying the conv?**
3. **In your own words:** explain why skip connections help preserve spatial accuracy in temperature maps.

*Your answers here:*


---
## Part F: Mini U-Net Assembly (3 min)

Now connect the MKB → Encoder → Bottleneck → Decoder into a single forward pass.


In [ ]:
# ── Part F: Assemble the Mini U-Net ─────────────────────────────────────────

class MiniUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.mkb     = MultiKernelBlock(in_channels=4, out_channels=8)   # 4→24ch
        # After MKB we have 24 channels; encoder expects 4 — add a channel adapter
        self.adapter = nn.Conv2d(24, 4, kernel_size=1)
        self.encoder = MiniEncoder()
        self.decoder = MiniDecoder()

    def forward(self, x):
        # Step 1 — Multi-Kernel Block
        mkb_feat  = self.mkb(x)           # (B, 24, 19, 19)
        adapted   = self.adapter(mkb_feat) # (B,  4, 19, 19)

        # Step 2 — Encoder
        e1, e2, bottleneck = self.encoder(________)   # fill in

        # Step 3 — Decoder with skip connection
        out = self.decoder(________, ________)         # fill in: (bottleneck, skip)

        return out

# ── Full forward pass ─────────────────────────────────────────────────────────
mini_unet = MiniUNet()

with torch.no_grad():
    unet_out = mini_unet(x_test)

print("Mini U-Net output shape:", unet_out.shape)
print("  → Spatial output matches input grid:", unet_out.shape[-2:], "≈", x_test.shape[-2:])

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(t2m_arr[0], cmap='RdYlBu_r', origin='lower')
axes[0].set_title('Actual T2M — Jan 1980')
axes[1].imshow(unet_out[0, 0].detach().numpy(), cmap='RdYlBu_r', origin='lower')
axes[1].set_title('Mini U-Net Output (untrained)')
for ax in axes:
    ax.set_xlabel('Lon index'); ax.set_ylabel('Lat index')
plt.suptitle('Q1 Complete — Spatial Pipeline Check', fontsize=12)
plt.tight_layout(); plt.show()
print("\n✅ Question 1 complete! The untrained U-Net produces a spatial temperature map.")


---
# ⏳ Question 2: Building the Temporal Learning Pipeline
**⏱ Estimated Time: 30 minutes**

In this question you implement the LSTM temporal module and assemble the full MKB–UNet–LSTM pipeline.


## Part A: Time Series Exploration (4 min)

Before building the LSTM, explore the T2M time series at a single grid cell to understand the temporal patterns the model must learn.


In [ ]:
# ── Part A: Extract and plot T2M time series at one grid cell ───────────────

# Pick the central grid cell
lat_idx = 9    # middle of the 19-point latitude axis
lon_idx = 9    # middle of the 19-point longitude axis

# Extract time series for each variable at this grid cell
t2m_series = t2m_arr[:, _______, _______]   # fill in lat_idx, lon_idx
bc_series  = bc_arr [:, lat_idx, lon_idx]
du_series  = du_arr [:, lat_idx, lon_idx]
su_series  = su_arr [:, lat_idx, lon_idx]

print(f"Grid cell: lat={lats[lat_idx]}°N, lon={lons[lon_idx]}°E")
print(f"T2M time series length: {len(t2m_series)} months")
print(f"T2M range: {t2m_series.min():.1f}°C  –  {t2m_series.max():.1f}°C")

# Plot the first 10 years (120 months)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

series = [t2m_series, bc_series, du_series, su_series]
labels = ['T2M (°C)', 'BC_AOD', 'DU_AOD', 'SU_AOD']
colors = ['firebrick', 'saddlebrown', 'goldenrod', 'steelblue']

for ax, s, lbl, col in zip(axes, series, labels, colors):
    ax.plot(s[:120], color=col, linewidth=1.2)
    ax.set_ylabel(lbl, fontsize=10)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Month index (0 = Jan 1980)')
plt.suptitle(f'Time Series at ({lats[lat_idx]}°N, {lons[lon_idx]}°E) — First 10 Years',
             fontsize=12)
plt.tight_layout(); plt.show()


### 📝 Interpretation — Part A

1. **What pattern do you see in T2M?** Is there a clear seasonal cycle?
2. **Do any aerosol variables show a seasonal pattern?** Which one is most pronounced?
3. **Why is this seasonality important** for an LSTM that must predict next month's temperature?

*Your answers here:*


---
## Part B: Create Sequences for LSTM (6 min)

LSTMs require data in the form:  
`(batch, sequence_length, features)` → predict the next value.

We use a sliding window: given `seq_len` past months of all variables, predict the next month's T2M.


In [ ]:
# ── Part B: Complete the sequence creation function ──────────────────────────

def create_sequences(t2m, bc, du, su, seq_len=12):
    """
    Build sliding-window input/target pairs for the LSTM.

    For each time step t (starting from seq_len):
        X[i] = all 4 variables from t-seq_len to t-1  → shape: (seq_len, 4)
        y[i] = T2M at time t                           → shape: (1,)

    Parameters
    ----------
    t2m, bc, du, su : 1-D arrays of length T (single grid cell)
    seq_len         : number of past months to use as input

    Returns
    -------
    X : np.ndarray, shape (N, seq_len, 4)
    y : np.ndarray, shape (N,)
    """
    X, y = [], []

    for t in range(seq_len, len(t2m)):
        # Stack the 4 variables for the window t-seq_len : t
        window = np.stack([
            t2m[t - seq_len : _______],   # fill in end index
            bc [t - seq_len : t],
            du [t - seq_len : t],
            su [t - seq_len : t]
        ], axis=_______)                  # fill in axis (each row = one month)

        X.append(window)
        y.append(_______)                 # fill in: target = T2M at time t

    return np.array(X), np.array(y)

# ── Build sequences ───────────────────────────────────────────────────────────
SEQ_LEN = 12   # use 12 months of history to predict the next month

X_seq, y_seq = create_sequences(t2m_series, bc_series, du_series, su_series,
                                 seq_len=SEQ_LEN)

print(f"X_seq shape: {X_seq.shape}  → (n_samples, seq_len, n_features)")
print(f"y_seq shape: {y_seq.shape}  → (n_samples,)")
print(f"\nFirst window input (first 3 time steps):")
print(X_seq[0, :3, :])
print(f"\nCorresponding target T2M: {y_seq[0]:.2f} °C")


In [ ]:
# ── Visualize a few sequences ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, ax in enumerate([10, 50, 100]):
    ax.plot(range(SEQ_LEN), X_seq[idx, :, 0], 'r-o', markersize=4,
            label='T2M input')
    ax.axvline(SEQ_LEN - 1, color='gray', linestyle='--', linewidth=1)
    ax.scatter(SEQ_LEN, y_seq[idx], color='darkred', zorder=5,
               s=80, label='Target (next month)')
    ax.set_title(f'Sample {idx}: months {idx}–{idx+SEQ_LEN}')
    ax.set_xlabel('Month in window')
    ax.set_ylabel('T2M (°C)')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('LSTM Input Sequences (red line = input, dot = target)', fontsize=11)
plt.tight_layout(); plt.show()


### 📝 Interpretation — Part B

1. **What does one row of `X_seq` represent?** (be specific about time and variables)
2. **Why do we use `seq_len=12`?** What climate phenomenon does this capture?
3. **If you increased `seq_len` to 24**, what additional patterns could the LSTM learn?

*Your answers here:*


---
## Part C: Implement a Simple LSTM (5 min)

Now build a single-layer LSTM that reads the 12-month sequences and predicts next month's T2M.


In [ ]:
# ── Part C: Complete the LSTM model ─────────────────────────────────────────

class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = _______,    # fill in: number of input features per time step
            hidden_size = _______,    # fill in: size of hidden state (try 64)
            num_layers  = 1,
            batch_first = True        # input shape: (batch, seq_len, features)
        )
        # Fully connected layer: hidden_size → output_size
        self.fc = nn.Linear(_______, output_size)   # fill in

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        lstm_out, (h_n, c_n) = self.lstm(x)

        # Use only the last time step's output for prediction
        last_out = lstm_out[:, -1, :]     # shape: (batch, hidden_size)

        prediction = self.fc(_______)     # fill in
        return prediction.squeeze(-1)

# ── Test with dummy input ─────────────────────────────────────────────────────
model_lstm = SimpleLSTM(input_size=4, hidden_size=64)
print("LSTM model architecture:")
print(model_lstm)

dummy_x = torch.randn(8, SEQ_LEN, 4)   # batch=8, seq=12, features=4
with torch.no_grad():
    dummy_out = model_lstm(dummy_x)

print(f"\nDummy input  shape: {dummy_x.shape}")
print(f"Dummy output shape: {dummy_out.shape}  ✅ (one prediction per batch element)")


### 📝 Interpretation — Part C

1. **Why do we use `lstm_out[:, -1, :]` (the last time step) instead of the full output?**
2. **What does `hidden_size=64` control?** What happens if you make it larger or smaller?
3. **The LSTM has gates (input, forget, output).** In your own words, what does the *forget gate* do for a climate time series?

*Your answers here:*


---
## Part D: Connect U-Net Spatial Features to LSTM (5 min)

In the full MKB–UNet–LSTM, the U-Net produces a **spatial feature map at each time step**.  
The LSTM then reads those feature maps as a sequence, learning how they evolve over time.

Here we simulate that pipeline using pre-extracted U-Net features.


In [ ]:
# ── Part D: Extract U-Net spatial features across time ───────────────────────
# Simulate the full dataset as a tensor: (T, 4, 19, 19)
all_data = np.stack([bc_arr[:, lat_idx, :].reshape(-1, 1, n_lon).repeat(n_lat, axis=1),
                     du_arr[:, lat_idx, :].reshape(-1, 1, n_lon).repeat(n_lat, axis=1),
                     su_arr[:, lat_idx, :].reshape(-1, 1, n_lon).repeat(n_lat, axis=1),
                     t2m_arr[:, lat_idx, :].reshape(-1, 1, n_lon).repeat(n_lat, axis=1)],
                    axis=1).astype(np.float32)

# Normalize each channel to [0, 1]
for ch in range(4):
    mn, mx = all_data[:, ch].min(), all_data[:, ch].max()
    all_data[:, ch] = (all_data[:, ch] - mn) / (mx - mn + 1e-8)

data_tensor = torch.tensor(all_data)   # (528, 4, 19, 19)
print("Full data tensor shape:", data_tensor.shape)

# Extract spatial feature vector from U-Net for each time step
unet_features = []
mini_unet.eval()
with torch.no_grad():
    for t in range(len(data_tensor)):
        x_t   = data_tensor[t].unsqueeze(0)          # (1, 4, 19, 19)
        feat  = mini_unet(x_t)                        # (1, 1, ~9, ~9)
        # Flatten spatial dimensions into a feature vector
        vec   = feat.view(1, -1)                      # (1, n_spatial_features)
        unet_features.append(vec.squeeze(0).numpy())

unet_features = np.array(unet_features)   # (528, n_spatial_features)
print("U-Net feature matrix shape:", unet_features.shape)
n_feat = unet_features.shape[1]


In [ ]:
# ── Build LSTM sequences from U-Net features ─────────────────────────────────
def create_feat_sequences(features, targets, seq_len=12):
    """Same as create_sequences but using U-Net feature vectors as inputs."""
    X, y = [], []
    for t in range(seq_len, len(targets)):
        X.append(features[t - seq_len : _______])   # fill in
        y.append(targets[t])
    return np.array(X), np.array(y)

t2m_full = t2m_arr[:, lat_idx, lon_idx]     # full T2M at the chosen grid cell

X_feat, y_feat = create_feat_sequences(unet_features, t2m_full, seq_len=SEQ_LEN)

print(f"X_feat shape: {X_feat.shape}  → (n_samples, seq_len, n_unet_features)")
print(f"y_feat shape: {y_feat.shape}")

# Reshape for LSTM: (batch, seq_len, features)
X_feat_t = torch.tensor(X_feat, dtype=torch.float32)
y_feat_t = torch.tensor(y_feat, dtype=torch.float32)

# Test forward pass through LSTM with U-Net features
lstm_feat = SimpleLSTM(input_size=_______, hidden_size=32)   # fill in n_feat
with torch.no_grad():
    pred_sample = lstm_feat(X_feat_t[:4])

print(f"\nLSTM(U-Net features) output shape: {pred_sample.shape}  ✅")


### 📝 Interpretation — Part D

1. **Why do we flatten the U-Net output** before feeding it to the LSTM?
2. **What information do the U-Net features carry** that raw T2M values alone would not?
3. **Sketch (in words) the data flow** from a raw climate CSV to an LSTM prediction in this pipeline.

*Your answers here:*


---
## Part E: End-to-End MKB–UNet–LSTM (5 min)

Assemble the complete architecture. Only **5–8 lines** need completing.


In [ ]:
# ── Part E: Complete the End-to-End Model ───────────────────────────────────

class MKB_UNet_LSTM(nn.Module):
    def __init__(self, spatial_feat_size, hidden_size=64, seq_len=12):
        super().__init__()
        self.seq_len = seq_len

        # Spatial branch
        self.mkb     = MultiKernelBlock(in_channels=4, out_channels=8)
        self.adapter = nn.Conv2d(24, 4, kernel_size=1)
        self.encoder = MiniEncoder()
        self.decoder = MiniDecoder()

        # Temporal branch
        self.lstm    = nn.LSTM(input_size=spatial_feat_size,
                               hidden_size=_______, batch_first=True)   # fill in

        # Output head
        self.fc      = nn.Linear(_______, 1)                            # fill in

    def forward(self, x_seq):
        """
        x_seq : (batch, seq_len, 4, H, W)  — a sequence of climate maps
        """
        B, T, C, H, W = x_seq.shape

        # ── Process each time step through MKB + U-Net ──────────────────────
        spatial_feats = []
        for t in range(T):
            x_t   = x_seq[:, t, :, :, :]               # (B, 4, H, W)
            mkb_f = self.mkb(x_t)                       # (B, 24, H, W)
            ada_f = self.adapter(mkb_f)                 # (B,  4, H, W)
            _, _, bottle = self.encoder(ada_f)          # bottleneck
            dec_f = self.decoder(bottle, _______        # fill in: which encoder output?
                                 )
            flat  = dec_f.view(B, -1)                   # (B, spatial_feat_size)
            spatial_feats.append(flat)

        # Stack into (B, T, spatial_feat_size) for the LSTM
        feats_seq = torch.stack(_______, dim=1)          # fill in

        # ── LSTM temporal learning ───────────────────────────────────────────
        lstm_out, _ = self.lstm(feats_seq)
        last        = lstm_out[:, -1, :]                 # last time step

        # ── Prediction head ──────────────────────────────────────────────────
        out = self.fc(_______)                           # fill in
        return out.squeeze(-1)

# ── Compute spatial feature size from a forward pass ─────────────────────────
with torch.no_grad():
    _dummy = mini_unet(x_test)
    _feat_size = _dummy.view(1, -1).shape[1]
print(f"Spatial feature size (from U-Net output): {_feat_size}")

full_model = MKB_UNet_LSTM(spatial_feat_size=_feat_size, hidden_size=64,
                            seq_len=SEQ_LEN)
print("\nFull MKB–UNet–LSTM model ready ✅")
total_params = sum(p.numel() for p in full_model.parameters())
print(f"Total trainable parameters: {total_params:,}")


---
## Part F: Prediction and Visualization (5 min)

Train a lightweight version (SimpleLSTM on U-Net features) and evaluate its predictions.


In [ ]:
# ── Part F: Train SimpleLSTM on extracted U-Net features ────────────────────

# Normalize targets
t2m_mean = float(y_feat_t.mean())
t2m_std  = float(y_feat_t.std())
y_norm   = (y_feat_t - t2m_mean) / t2m_std

# Train / test split (80/20)
split    = int(0.8 * len(X_feat_t))
X_train, X_test_data = X_feat_t[:split],  X_feat_t[split:]
y_train, y_test_data = y_norm[:split],     y_norm[split:]

# ── Complete the training loop ────────────────────────────────────────────────
model_train = SimpleLSTM(input_size=n_feat, hidden_size=64)
optimizer   = torch.optim.Adam(model_train.parameters(), lr=1e-3)
loss_fn     = nn.MSELoss()

EPOCHS   = 30
BATCH    = 32
dataset  = TensorDataset(X_train, y_train)
loader   = DataLoader(dataset, batch_size=BATCH, shuffle=True)

train_losses = []

for epoch in range(EPOCHS):
    model_train.train()
    epoch_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        preds = model_train(_______)              # fill in
        loss  = loss_fn(preds, _______)           # fill in
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(loader))
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  |  Loss: {train_losses[-1]:.4f}")


In [ ]:
# ── Evaluate and visualize predictions ───────────────────────────────────────
model_train.eval()
with torch.no_grad():
    y_pred_norm = model_train(X_test_data).numpy()

# Denormalize
y_pred_actual = y_pred_norm * t2m_std + t2m_mean
y_true_actual = y_test_data.numpy() * t2m_std + t2m_mean

# Compute RMSE
rmse = np.sqrt(np.mean((y_pred_actual - y_true_actual) ** 2))
print(f"Test RMSE: {rmse:.2f} °C")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Time series comparison
axes[0].plot(y_true_actual[:60], label='Actual T2M', color='firebrick', linewidth=1.5)
axes[0].plot(y_pred_actual[:60], label='Predicted T2M', color='steelblue',
             linewidth=1.5, linestyle='--')
axes[0].set_xlabel('Test month index')
axes[0].set_ylabel('T2M (°C)')
axes[0].set_title(f'Actual vs Predicted T2M (first 60 test months)\nRMSE = {rmse:.2f} °C')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Scatter plot
axes[1].scatter(y_true_actual, y_pred_actual, alpha=0.4, s=20, color='purple')
mn_v = min(y_true_actual.min(), y_pred_actual.min()) - 1
mx_v = max(y_true_actual.max(), y_pred_actual.max()) + 1
axes[1].plot([mn_v, mx_v], [mn_v, mx_v], 'k--', linewidth=1)
axes[1].set_xlabel('Actual T2M (°C)')
axes[1].set_ylabel('Predicted T2M (°C)')
axes[1].set_title('Scatter: Actual vs Predicted')
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Training loss curve
plt.figure(figsize=(7, 3))
plt.plot(train_losses, color='darkgreen', linewidth=1.5)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('Training Loss Curve'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 📝 Interpretation — Part F

1. **What is your RMSE?** Is predicting within ±X°C accurate enough for climate applications?
2. **What does the scatter plot tell you** about the model's behaviour across the temperature range?
3. **The model uses only one grid cell's time series.** How might performance change if you used the full 19×19 spatial grid for each LSTM step?
4. **Name one improvement** you would make to this pipeline if you had more compute time.

*Your answers here:*


---
## 🎉 Lab Complete!

You have successfully implemented:

| Component | What you built |
|-----------|----------------|
| **MKB** | Multi-scale convolutions (1×1, 3×3, 5×5) fused via concatenation |
| **U-Net Encoder** | Conv + MaxPool spatial downsampling |
| **U-Net Decoder** | Upsample + skip connections for spatial recovery |
| **LSTM** | Sequence-to-one temporal prediction |
| **End-to-End** | MKB → U-Net → LSTM climate forecast |

> **Key Takeaway:** Each block has a specific role — the MKB extracts multi-scale aerosol patterns, the U-Net preserves spatial context, and the LSTM learns how those patterns evolve month-to-month to predict temperature.
